In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score

df = pd.read_csv("[머신러닝프로젝트][가공데이터].csv", encoding='utf-8-sig')


# 2. 계절 파생 변수 생성 함수
def get_season(month):
    if month in [3, 4, 5]: return '봄'
    elif month in [6, 7, 8]: return '여름'
    elif month in [9, 10, 11]: return '가을'
    else: return '겨울'

df['계절'] = df['월'].apply(get_season)
seasons = ['봄', '여름', '가을', '겨울']

print("[최종 모델] 기상 지표 + 주말여부 결합 예측 성능 평가\n")

# 3. 4계절 순회하며 맞춤형 특성(Feature) 적용 및 모델 튜닝
for season in seasons:
    print(f"{season}철 전력 예측 모델 튜닝 시작")
    
    season_df = df[df['계절'] == season]
    
    if season == '여름':
        features = ['평균 상대습도(%)', '평균 풍속(m/s)', '일강수량(mm)', '불쾌지수', '주말여부']
    elif season == '겨울':
        features = ['평균 상대습도(%)', '평균 풍속(m/s)', '일강수량(mm)', '일 최심신적설(cm)', '체감온도', '주말여부']
    else: # 봄, 가을
        features = ['평균기온(°C)', '평균 상대습도(%)', '평균 풍속(m/s)', '일강수량(mm)', '주말여부']

    # 독립변수(X)와 종속변수(y)
    X = season_df[features]
    y = season_df['최대전력(MW)']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # ------------------------------------------
    # 모델 1: Random Forest 파이프라인 튜닝
    # ------------------------------------------
    rf_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(random_state=42))
    ])
    
    rf_param_grid = {
        'rf__n_estimators': [50, 100, 200],
        'rf__max_depth': [None, 10, 20]
    }
    
    rf_grid = GridSearchCV(rf_pipe, param_grid=rf_param_grid, cv=3, scoring='r2', n_jobs=-1)
    rf_grid.fit(X_train, y_train)
    rf_test_score = r2_score(y_test, rf_grid.predict(X_test))

    print(f"\n[Random Forest]")
    print(f" - 최적 파라미터: {rf_grid.best_params_}")
    print(f" - 최종 실전 예측력 (R² Score): {rf_test_score:.2f}")

    # ------------------------------------------
    # 모델 2: Gradient Boosting 파이프라인 튜닝
    # ------------------------------------------
    gb_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('gb', GradientBoostingRegressor(random_state=42))
    ])
    
    gb_param_grid = {
        'gb__n_estimators': [100, 200],
        'gb__learning_rate': [0.01, 0.1, 0.2],
        'gb__max_depth': [3, 5, 7]
    }
    
    gb_grid = GridSearchCV(gb_pipe, param_grid=gb_param_grid, cv=3, scoring='r2', n_jobs=-1)
    gb_grid.fit(X_train, y_train)
    gb_test_score = r2_score(y_test, gb_grid.predict(X_test))

    print(f"\n[Gradient Boosting]")
    print(f" - 최적 파라미터: {gb_grid.best_params_}")
    print(f" - 최종 실전 예측력 (R² Score): {gb_test_score:.2f}\n")

[최종 모델] 기상 지표 + 주말여부 결합 예측 성능 평가

봄철 전력 예측 모델 튜닝 시작

[Random Forest]
 - 최적 파라미터: {'rf__max_depth': 10, 'rf__n_estimators': 200}
 - 최종 실전 예측력 (R² Score): 0.80

[Gradient Boosting]
 - 최적 파라미터: {'gb__learning_rate': 0.2, 'gb__max_depth': 3, 'gb__n_estimators': 100}
 - 최종 실전 예측력 (R² Score): 0.75

여름철 전력 예측 모델 튜닝 시작

[Random Forest]
 - 최적 파라미터: {'rf__max_depth': None, 'rf__n_estimators': 100}
 - 최종 실전 예측력 (R² Score): 0.90

[Gradient Boosting]
 - 최적 파라미터: {'gb__learning_rate': 0.1, 'gb__max_depth': 3, 'gb__n_estimators': 100}
 - 최종 실전 예측력 (R² Score): 0.88

가을철 전력 예측 모델 튜닝 시작

[Random Forest]
 - 최적 파라미터: {'rf__max_depth': None, 'rf__n_estimators': 200}
 - 최종 실전 예측력 (R² Score): 0.70

[Gradient Boosting]
 - 최적 파라미터: {'gb__learning_rate': 0.01, 'gb__max_depth': 3, 'gb__n_estimators': 200}
 - 최종 실전 예측력 (R² Score): 0.67

겨울철 전력 예측 모델 튜닝 시작

[Random Forest]
 - 최적 파라미터: {'rf__max_depth': None, 'rf__n_estimators': 100}
 - 최종 실전 예측력 (R² Score): 0.54

[Gradient Boosting]
 - 최적 파라미터: {'gb__learning_rate

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 1. 데이터를 불러오고 계절 파생 변수를 생성하는 과정은 동일하게 진행

seasons = ['봄', '여름', '가을', '겨울']

# 2. 계절별로 학습이 완료된 '최고의 모델'들을 담아둘 딕셔너리 생성
final_models = {}

for season in seasons:
    season_df = df[df['계절'] == season]
    
    # 특성(Feature) 분리
    if season == '여름':
        features = ['평균 상대습도(%)', '평균 풍속(m/s)', '일강수량(mm)', '불쾌지수', '주말여부']
    elif season == '겨울':
        features = ['평균 상대습도(%)', '평균 풍속(m/s)', '일강수량(mm)', '일 최심신적설(cm)', '체감온도', '주말여부']
    else: 
        features = ['평균기온(°C)', '평균 상대습도(%)', '평균 풍속(m/s)', '일강수량(mm)', '주말여부']

    X = season_df[features]
    y = season_df['최대전력(MW)']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # ★ 3. 조건문을 통해 계절별 1등 모델을 다르게 지정하고 학습
    if season in ['봄', '여름', '가을']:
        # Random Forest가 1등인 계절 (GridSearchCV로 찾은 최적 파라미터 입력)
        model = Pipeline([
            ('scaler', StandardScaler()),
            # 예시: 여름철 최적 파라미터가 n_estimators=100, max_depth=None 이었다면 아래에 입력
            ('rf', RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42)) 
        ])
    else:
        # 겨울 (Gradient Boosting이 1등인 계절)
        model = Pipeline([
            ('scaler', StandardScaler()),
            # 예시: 겨울철 최적 파라미터가 learning_rate=0.2, max_depth=3, n_estimators=100 이었다면 아래에 입력
            ('gb', GradientBoostingRegressor(learning_rate=0.2, max_depth=3, n_estimators=100, random_state=42))
        ])
    
    # 모델 학습
    model.fit(X_train, y_train)
    
    # 학습된 모델을 딕셔너리에 저장
    final_models[season] = model
    print(f"{season}철 최적 모델 학습 및 저장 완료!")

# ==========================================
# 4. 미래의 새로운 데이터를 예측할 때 (실무 적용 함수)
# ==========================================
def predict_tomorrow_power(input_data_df):
    """
    내일의 기상 예보 데이터를 넣으면, 해당 계절에 맞는 1등 모델을 자동으로 꺼내와 예측하는 함수
    """
    # 입력된 데이터의 계절 확인
    target_season = input_data_df['계절'].iloc[0]
    
    # 해당 계절에 맞는 특성(Feature)만 추출
    if target_season == '여름':
        features = ['평균 상대습도(%)', '평균 풍속(m/s)', '일강수량(mm)', '불쾌지수', '주말여부']
    elif target_season == '겨울':
        features = ['평균 상대습도(%)', '평균 풍속(m/s)', '일강수량(mm)', '일 최심신적설(cm)', '체감온도', '주말여부']
    else: 
        features = ['평균기온(°C)', '평균 상대습도(%)', '평균 풍속(m/s)', '일강수량(mm)', '주말여부']
        
    X_new = input_data_df[features]
    
    # 저장해둔 1등 모델을 꺼내서 예측
    best_model = final_models[target_season]
    prediction = best_model.predict(X_new)[0]
    
    return prediction

봄철 최적 모델 학습 및 저장 완료!
여름철 최적 모델 학습 및 저장 완료!
가을철 최적 모델 학습 및 저장 완료!
겨울철 최적 모델 학습 및 저장 완료!


In [ ]:
import pandas as pd

# ==========================================
# 시나리오 1: 한여름 폭염 경보가 내린 평일 (8월)
# ==========================================
# 예보: 습도 85%, 바람 거의 없음, 불쾌지수 82(매우 높음), 평일(주말여부=0)
summer_scenario = pd.DataFrame({
    '계절': ['여름'],
    '평균 상대습도(%)': [85.0],
    '평균 풍속(m/s)': [1.2],
    '일강수량(mm)': [0.0],
    '불쾌지수': [82.5], 
    '주말여부': [0]     
})

# ==========================================
# 시나리오 2: 칼바람 불고 눈 내리는 한파 평일 (1월)
# ==========================================
# 예보: 습도 40%, 강풍 6.5m/s, 눈 3cm, 체감온도 영하 14도, 평일(주말여부=0)
winter_scenario = pd.DataFrame({
    '계절': ['겨울'],
    '평균 상대습도(%)': [40.0],
    '평균 풍속(m/s)': [6.5], 
    '일강수량(mm)': [0.0],
    '일 최심신적설(cm)': [3.0], 
    '체감온도': [-14.0],       
    '주말여부': [0]            
})


summer_pred = predict_tomorrow_power(summer_scenario)
winter_pred = predict_tomorrow_power(winter_scenario)

print("[전력거래소 내일의 최대전력 예측 보고서] \n")
print(f"폭염 시나리오: 내일 예상 최대전력은 {summer_pred:,.0f} MW 입니다.")
print("-> (내부 작동 알고리즘: 여름철 1등 모델 'Random Forest' 자동 호출)\n")

print(f"❄️ 한파 시나리오: 내일 예상 최대전력은 {winter_pred:,.0f} MW 입니다.")
print("   -> (내부 작동 알고리즘: 겨울철 1등 모델 'Gradient Boosting' 자동 호출)")

[전력거래소 내일의 최대전력 예측 보고서] 

폭염 시나리오: 내일 예상 최대전력은 89,442 MW 입니다.
-> (내부 작동 알고리즘: 여름철 1등 모델 'Random Forest' 자동 호출)

❄️ 한파 시나리오: 내일 예상 최대전력은 83,804 MW 입니다.
   -> (내부 작동 알고리즘: 겨울철 1등 모델 'Gradient Boosting' 자동 호출)


In [ ]:
import joblib
import json

# 1. 4계절 모델이 모두 담긴 딕셔너리를 하나의 파일로 통째로 저장
joblib.dump(final_models, 'power_models.joblib')
print("✅ 4계절 예측 모델이 'power_models.joblib'로 통합 저장되었습니다.")

# 2. 계절별 모델 성능 정보(JSON) 만들기
model_info = {
    "봄": {"r2_score": 0.796, "model_type": "Random Forest"},
    "여름": {"r2_score": 0.905, "model_type": "Random Forest"},
    "가을": {"r2_score": 0.698, "model_type": "Random Forest"},
    "겨울": {"r2_score": 0.577, "model_type": "Gradient Boosting"}
}

with open('power_model_info.json', 'w', encoding='utf-8') as f:
    json.dump(model_info, f, ensure_ascii=False, indent=2)
print("✅ 모델 성능 정보가 'power_model_info.json'으로 저장되었습니다.")

✅ 4계절 예측 모델이 'power_models.joblib'로 통합 저장되었습니다.
✅ 모델 성능 정보가 'power_model_info.json'으로 저장되었습니다.
